# Bayesian Optimisation: ERK Oscillation Frequency (BoTorch)

Uses the `BOptBoTorch` inter-phase agent with **batch acquisition** to find
the optimal optogenetic stimulation parameters that maximise the fraction
of oscillating cells (ERK-KTR cytoplasmic/nuclear ratio oscillations).

**This notebook is identical to `bo_erk_oscillation_test.ipynb` except**
**it uses BoTorch (PyTorch/GPyTorch) instead of gpax (JAX/NumPyro).**

**GP model:** SingleTaskGP with Matern 5/2 kernel (MLE via L-BFGS,
built-in Normalize + Standardize transforms). No JAX dependency.

**Steerable parameters (BO optimises):**
- `stim_exposure` -- base stimulation light exposure in ms (50--500, step 25)
- `ramp` -- per-frame exposure increment in ms (0--50, step 5).

**Covariates (observed, not controlled):**
- `n_cells` -- number of tracked cells in the FOV
- `optortk_expression` -- mean optocheck (mCitrine) intensity per FOV
- `baseline_cnr` -- mean pre-stim ERK activity per FOV

**Objective:** maximise `frac_oscillating` -- fraction of cells classified as
oscillating per FOV, using a pre-trained sliding-window XGBoost classifier.

See `bo_erk_oscillation_test.ipynb` for full documentation of the
phased acquisition, FOV finding, initial sampling, and batch BO strategies.

In [ ]:
import os
import time
import logging
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from faro.core.data_structures import (
    PowerChannel,
    SegmentationMethod,
)
from faro.core.controller import Controller
from faro.core.pipeline import ImageProcessingPipeline
from faro.agents.bo_optimization import (
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)
from faro.agents.bo_botorch_oscillation import OscillationBOBoTorch
import faro.core.utils as utils

## Microscope & pipeline setup

**Microscope:** Jungfrau (no DMD -- full-FOV stimulation).

**Channels:**
- Imaging: miRFP (nuclear marker) + mScarlet3 (ERK-KTR reporter)
- Stimulation: CyanStim (optogenetic activation, power 10)
- Optocheck: mCitrine (verify optoRTK expression, last frame only)

**Pipeline:** CellposeV4 segmentation -> ERK-KTR FE -> Trackpy tracking
-> OptoCheckFE on reference frames. Full-FOV stimulation (no DMD).

In [ ]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")
mic.mmc.setProperty("TIPFSStatus", "State", "On")

In [ ]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc  # point the widget at our CMMCore instance
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

In [ ]:
# --- Experiment parameters ---
TIME_BETWEEN_TIMESTEPS = 60  # seconds (1 frame/min)

# --- Phase layout (frames; 1 frame == 1 minute at TIME_BETWEEN_TIMESTEPS=60) ---
# 10 min baseline -> 60 min stimulation -> 20 min recovery (cool-down).
N_FRAMES_BASELINE = 10
N_FRAMES_STIM = 60
N_FRAMES_RECOVERY = 20

# Derived (don't edit -- adjust the three above instead).
N_FRAMES = N_FRAMES_BASELINE + N_FRAMES_STIM + N_FRAMES_RECOVERY  # 90
FIRST_FRAME_STIM = N_FRAMES_BASELINE  # 10
LAST_FRAME_STIM = FIRST_FRAME_STIM + N_FRAMES_STIM  # 70

# --- Plate calibration ---
PLATE_CALIBRATION_PATH = r"./calib_plate_96.json"  # <-- UPDATE

# --- Wells ---
WELLS = []
for i, row in enumerate("ABCDEFGH"):
    cols = range(2, 13) if i % 2 == 0 else range(12, 1, -1)
    WELLS.extend(f"{row}{c}" for c in cols)

WELLS = WELLS[4:]

# --- FOV finder knobs ---
FOV_BORDER_UM = 1500.0
FOV_MIN_DISTANCE_UM = 1000.0
FOV_MIN_CELLS = 35
FOV_N_CANDIDATES_PER_WELL = 6

# --- Phased FOV layout ---
N_WELLS_PER_PHASE = 6
FOVS_PER_WELL = 3
N_FOVS = N_WELLS_PER_PHASE * FOVS_PER_WELL  # 18
N_CONDITIONS_PER_ITER = 3
FOVS_PER_CONDITION = N_FOVS // N_CONDITIONS_PER_ITER  # 6
N_PHASES = 14

assert (
    len(WELLS) >= N_PHASES * N_WELLS_PER_PHASE
), f"Need at least {N_PHASES * N_WELLS_PER_PHASE} wells, got {len(WELLS)}"
assert (
    len(WELLS) % N_WELLS_PER_PHASE == 0
), f"Number of wells ({len(WELLS)}) must be divisible by N_WELLS_PER_PHASE ({N_WELLS_PER_PHASE})"

# --- Storage ---
base_path = "E:\\Alex"
experiment_name = "2026-04-10_bo_erk_oscillation_botorch"
path = os.path.join(base_path, experiment_name)

# --- Stimulation channel ---
stim_channel = PowerChannel(
    config="CyanStim",
    exposure=100,
    group="TTL_ERK",
    power=10,
)

# --- Imaging channels ---
imaging_channels = (
    PowerChannel(config="miRFP", exposure=150, group="TTL_ERK", power=95),
    PowerChannel(config="mScarlet3", exposure=150, group="TTL_ERK", power=95),
)

# --- Optocheck channel ---
optocheck_channel = PowerChannel(
    config="mCitrine",
    exposure=600,
    group="TTL_ERK",
    power=95,
)

In [ ]:
WELLS

In [ ]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

segmentator = CellposeV4(
    custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
    min_size=100,
)

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            name="labels",
            segmentation_class=segmentator,
            use_channel=0,
            save_tracked=True,
        )
    ],
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimWholeFOV(),
    feature_extractor_ref=OptoCheckFE(used_mask="labels"),
)

from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)

## FOV selection (per-phase, automated)

Same as the gpax notebook: `FOVFinderAgent` picks fresh FOVs before every
phase. See `bo_erk_oscillation_test.ipynb` for full documentation.

## Oscillation classifier

Load the pre-trained sliding-window oscillation classifier (same as gpax
notebook -- the classifier is backend-independent).

In [ ]:
import joblib

# --- Load pre-trained oscillation classifier ---
CLASSIFIER_PATH = r"./oscillation_model_60min.joblib"  # <-- UPDATE THIS PATH

model_data = joblib.load(CLASSIFIER_PATH)
osc_clf = model_data["clf"]
osc_scaler = model_data["scaler"]
osc_feature_cols = model_data["feature_cols"]
osc_cfg = model_data["config"]
osc_cfg["window_size"] = model_data["window_size"]
osc_cfg["window_step"] = model_data["window_step"]

# Import the predict_trace function from the classifier script
_classifier_script = "./apply_oscillation_classifier_v2.py"

_spec = importlib.util.spec_from_file_location("osc_classifier", _classifier_script)
_osc_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_osc_module)
predict_trace = _osc_module.predict_trace

# --- Oscillation thresholds ---
MIN_OSC_PROBABILITY = 0.93
MIN_CONSECUTIVE_WINDOWS = 3
MIN_FFT_AMPLITUDE = 0.3

print(f"Loaded oscillation classifier from {CLASSIFIER_PATH}")
print(f"  Window: {osc_cfg['window_size']} steps, stride {osc_cfg['window_step']}")
print(
    f"  Thresholds: prob >= {MIN_OSC_PROBABILITY}, "
    f"consecutive >= {MIN_CONSECUTIVE_WINDOWS}, "
    f"fft_amplitude >= {MIN_FFT_AMPLITUDE}"
)

## Batch BO agent (BoTorch)

The :class:`OscillationBOBoTorch` class is a plug-and-play replacement for
:class:`OscillationBO`. It inherits all experiment-specific logic (event
creation, oscillation classification, live plotting, checkpointing) from
:class:`OscillationBO` and swaps the GP backend to BoTorch:

- **GP model:** `SingleTaskGP` with Matern 5/2 kernel, MLE hyperparameters
  (L-BFGS), built-in `Normalize` + `Standardize` transforms.
- **Acquisition:** Closed-form EI / UCB with covariate marginalisation,
  EI-to-UCB fallback when acquisition is flat.
- **Batch BO:** Sequential greedy acquisition with local penalisation
  (inherited from `BOptGPAX`).
- **No JAX dependency** — uses PyTorch / GPyTorch only.

## Configure and run Bayesian Optimisation

**Parameter space:**
- `stim_exposure`: 50--500 ms (step 25) = 19 levels
- `ramp`: 0--50 ms/frame (step 5) = 11 levels
- Total grid: 19 x 11 = 209 candidate conditions

**Covariates:** `n_cells`, `optortk_expression`, and `baseline_cnr` are
observed per FOV and marginalised over during acquisition evaluation.

In [ ]:
from faro.agents import FOVFinderAgent, ComposedAgent

# --- BO parameters ---
bo_params = [
    BO_Parameter(name="stim_exposure", bounds=(50.0, 500.0), spacing=25.0),
    BO_Parameter(name="ramp", bounds=(0.0, 30.0), spacing=5.0),
]

# --- Covariates (observed, not controlled) ---
bo_covariates = [
    BO_Covariate(name="n_cells"),
    BO_Covariate(name="optortk_expression"),
    BO_Covariate(name="baseline_cnr"),
]

# --- Objective ---
bo_objective = BO_Objective(name="frac_oscillating", goal="maximize")

# --- Pre-phase agent: FOV finder ---
fov_finder = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=WELLS,
    wells_per_phase=N_WELLS_PER_PHASE,
    fovs_per_well=FOVS_PER_WELL,
    n_candidates_per_well=FOV_N_CANDIDATES_PER_WELL,
    border_um=FOV_BORDER_UM,
    min_distance_um=FOV_MIN_DISTANCE_UM,
    min_cells=FOV_MIN_CELLS,
    imaging_channels=(imaging_channels[0],),
    segmentator=segmentator,
    seg_channel_index=0,
    random_seed=42,
    verbose=False,
    z=None,
)
print(
    f"FOVFinder: {len(WELLS)} wells queued -> "
    f"{fov_finder.n_remaining_phases} phases @ {N_WELLS_PER_PHASE} wells/phase"
)

# --- BO agent (BoTorch backend) ---
# Only change vs gpax notebook: OscillationBOBoTorch instead of OscillationBO.
agent = OscillationBOBoTorch(
    storage_path=path,
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    imaging_channels=imaging_channels,
    stim_channel=stim_channel,
    optocheck_channel=optocheck_channel,
    osc_clf=osc_clf,
    osc_scaler=osc_scaler,
    osc_feature_cols=osc_feature_cols,
    osc_cfg=osc_cfg,
    osc_predict_fn=predict_trace,
    min_osc_probability=MIN_OSC_PROBABILITY,
    min_consecutive_windows=MIN_CONSECUTIVE_WINDOWS,
    min_fft_amplitude=MIN_FFT_AMPLITUDE,
    n_baseline_frames=N_FRAMES_BASELINE,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=N_PHASES,
    n_conditions_per_iter=N_CONDITIONS_PER_ITER,
    n_initial_phases=2,
    acquisition_function="ei",
    n_cov_samples=20,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=True,
)

# --- Composed agent ---
composed_agent = ComposedAgent(
    inner_agent=agent,
    pre_phase_agents=[fov_finder],
    n_phases=N_PHASES,
)

# --- Controller ---
ctrl = Controller(mic, pipeline, writer=writer, agent=composed_agent)

print(f"Parameter grid: {len(agent.x_total_linespace)} candidate conditions")
print(
    f"Phases: {N_PHASES}  ({agent.n_initial_phases} initial-spread + "
    f"{N_PHASES - agent.n_initial_phases} BO batches)"
)
print(
    f"Conditions per phase: {N_CONDITIONS_PER_ITER}  "
    f"FOVs per condition: {FOVS_PER_CONDITION}  Total FOVs/phase: {N_FOVS}"
)
print(f"Total FOV observations after {N_PHASES} phases: ~{N_PHASES * N_FOVS}")
print(f"\nGP backend: BoTorch (SingleTaskGP, Matern 5/2, MLE)")

In [ ]:
# --- Run the composed (FOV finder + BO) loop ---
# Each phase: FOVFinder picks 18 fresh FOVs (3 FOVs in 6 fresh wells), then
# OscillationBOBoTorch runs one batch BO iteration over those FOVs.  The GP
# accumulates observations across all phases.

sleep_time = 3600 * 12  # sleep until tomorrow morning
for t in range(0, sleep_time):
    time.sleep(1)


composed_agent.run()

In [ ]:
# Post-processing: merge per-FOV tracks
utils.generate_exp_data_from_tracks(path)

## Visualise results

In [ ]:
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Scatter: stim_exposure vs ramp, colored by frac_oscillating
    sc = axes[0].scatter(
        df["stim_exposure"],
        df["ramp"],
        c=df["frac_oscillating"],
        cmap="viridis",
        s=40,
        edgecolors="k",
        linewidths=0.5,
    )
    axes[0].set_xlabel("stim_exposure (ms)")
    axes[0].set_ylabel("ramp (ms/frame)")
    axes[0].set_title("Measured frac_oscillating per FOV")
    fig.colorbar(sc, ax=axes[0], label="frac_oscillating")

    # 2. Convergence
    if agent._iteration_means:
        means = np.array(agent._iteration_means)
        cumulative_best = np.maximum.accumulate(means)
        axes[1].plot(means, "o-", alpha=0.5, label="iteration mean")
        axes[1].plot(cumulative_best, "s-", color="red", label="cumulative best")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("frac_oscillating")
        axes[1].set_title("BO convergence")
        axes[1].legend()

    # 3. Per-condition aggregation
    cond_agg = (
        df.groupby(["stim_exposure", "ramp"])
        .agg(
            mean_frac=("frac_oscillating", "mean"),
            n_fovs=("frac_oscillating", "count"),
        )
        .reset_index()
    )
    sc2 = axes[2].scatter(
        cond_agg["stim_exposure"],
        cond_agg["ramp"],
        c=cond_agg["mean_frac"],
        s=cond_agg["n_fovs"] * 30,
        cmap="viridis",
        edgecolors="k",
        linewidths=0.5,
    )
    axes[2].set_xlabel("stim_exposure (ms)")
    axes[2].set_ylabel("ramp (ms/frame)")
    axes[2].set_title("Mean frac_oscillating per condition\n(size = n_fovs)")
    fig.colorbar(sc2, ax=axes[2], label="mean frac_oscillating")

    plt.tight_layout()
    plt.show()

    # Print best condition
    best_idx = cond_agg["mean_frac"].idxmax()
    best = cond_agg.loc[best_idx]
    print(f"\nBest condition (by mean across FOVs):")
    print(f"  stim_exposure = {best['stim_exposure']:.0f} ms")
    print(f"  ramp = {best['ramp']:.0f} ms/frame")
    print(f"  mean frac_oscillating = {best['mean_frac']:.4f}")
    print(f"  tested on {int(best['n_fovs'])} FOVs")
else:
    print("No data collected yet.")

In [ ]:
# GP-predicted landscape, marginalised over covariates.
# Uses BoTorch model.posterior() -- no gpax / JAX dependency.
import torch

if agent.model is not None and agent.x is not None:
    # Controllable-parameter grid
    exp_vals = np.linspace(50.0, 500.0, 20)
    ramp_vals = np.linspace(0.0, 50.0, 15)
    ctrl_grid = np.array([[e, r] for e in exp_vals for r in ramp_vals])
    n_grid = ctrl_grid.shape[0]

    # Joint covariate samples from df_results (correlations preserved).
    df = agent.df_results
    n_cov_samples = 100
    cov_vals_full = df[[c.name for c in agent.bo_covariates]].to_numpy(dtype=float)
    if cov_vals_full.shape[0] == 0:
        raise RuntimeError("No covariate observations available for marginalisation.")
    _rng = np.random.default_rng(0)
    idx = _rng.integers(0, cov_vals_full.shape[0], size=n_cov_samples)
    cov_samples = cov_vals_full[idx]  # (n_cov_samples, n_covariates), joint

    # Build (x, c) full input
    x_full = np.hstack(
        [
            np.repeat(ctrl_grid, n_cov_samples, axis=0),
            np.tile(cov_samples, (n_grid, 1)),
        ]
    )

    # Apply log transforms and predict with BoTorch
    x_transformed = agent._apply_log_transforms(x_full)
    X_pred = torch.tensor(x_transformed, dtype=torch.double)

    with torch.no_grad():
        posterior = agent.model.posterior(X_pred)
        y_pred = posterior.mean.squeeze(-1).numpy()

    # Marginalise over covariates
    y_pred_marg = y_pred.reshape(n_grid, n_cov_samples).mean(axis=1)
    Y_grid = y_pred_marg.reshape(len(exp_vals), len(ramp_vals))

    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    cf = ax.contourf(ramp_vals, exp_vals, Y_grid, levels=20, cmap="viridis")
    fig.colorbar(cf, ax=ax, label="predicted frac_oscillating (marginalised)")

    # Overlay measured points
    ax.scatter(
        df["ramp"],
        df["stim_exposure"],
        c=df["frac_oscillating"],
        cmap="viridis",
        s=30,
        edgecolors="white",
        linewidths=0.5,
        zorder=5,
    )

    # Mark GP optimum
    opt_idx = int(np.argmax(y_pred_marg))
    opt_exp = ctrl_grid[opt_idx, 0]
    opt_ramp = ctrl_grid[opt_idx, 1]
    ax.scatter(
        opt_ramp,
        opt_exp,
        c="red",
        s=200,
        marker="*",
        edgecolors="black",
        linewidths=2,
        zorder=10,
        label=f"GP optimum: exp={opt_exp:.0f}ms, ramp={opt_ramp:.0f}ms",
    )

    ax.set_xlabel("ramp (ms/frame)")
    ax.set_ylabel("stim_exposure (ms)")
    ax.set_title(
        "GP-predicted frac_oscillating landscape (BoTorch)\n"
        f"(marginalised over {n_cov_samples} joint covariate samples "
        f"from df_results)"
    )
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    print("GP-predicted optimum (marginalised over covariates):")
    print(f"  stim_exposure = {opt_exp:.0f} ms")
    print(f"  ramp = {opt_ramp:.0f} ms/frame")
    print(f"  predicted frac_oscillating = {float(y_pred_marg[opt_idx]):.4f}")

In [ ]:
# Covariate effects
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(df["n_cells"], df["frac_oscillating"], alpha=0.6, s=20)
    axes[0].set_xlabel("n_cells per FOV")
    axes[0].set_ylabel("frac_oscillating")
    axes[0].set_title("Cell density vs oscillation fraction")

    axes[1].scatter(df["optortk_expression"], df["frac_oscillating"], alpha=0.6, s=20)
    axes[1].set_xlabel("optoRTK expression (mCitrine intensity)")
    axes[1].set_ylabel("frac_oscillating")
    axes[1].set_title("optoRTK expression vs oscillation fraction")

    plt.tight_layout()
    plt.show()